# Starting with hrtfpykit.plots

{doc}`hrtfpykit.plots <../plots/index>` contains the plotting functions for loaded {class}`HRTF <hrtfpykit.hrtf.HRTF>` objects, HRTF comparisons, and spherical-harmonic diagnostics. Single-HRTF plots take one HRTF object as their first argument and visualize its current IR, TF, and source-grid state. Comparison plots take two or more HRTF objects and visualize shared positions, cue differences, and reconstruction quality.


## Download two SONICOM HRTF files

Comparison functions need at least two loaded HRTF objects. A {class}`SONICOM <hrtfpykit.datasets.SONICOM>` dataset object prepares two measured `SimpleFreeFieldHRIR` SOFA files: `P0001_FreeFieldComp_44kHz.sofa` and `P0002_FreeFieldComp_44kHz.sofa`.

The dataset root is relative, so the files are stored under `datasets/sonicom` inside the folder where the notebook is run. The selected download variant is measured, 44.1 kHz, `FreeFieldComp`, matching the default SONICOM HRTF variant used by hrtfpykit.


In [ ]:
from pathlib import Path

from hrtfpykit.datasets import SONICOM

# Define the local dataset root.
root = Path("datasets/sonicom")

# Keep this tutorial small: use only P0001 and P0002.
selected_subject_ids = ("P0001", "P0002")

# Download only the measured 44.1 kHz FreeFieldComp HRTF resources for P0001 and P0002.
SONICOM(
    root=root,
    download=True,
    download_resources="hrtf",
    download_hrtf_variant={
        "type": "measured",
        "sample_rate": 44100,
        "version": "FreeFieldComp",
    },
    download_subject_ids=selected_subject_ids,
    subject_ids=selected_subject_ids,
    verify_checksum=True,
)

# Build paths to the downloaded SOFA files.
pp1_path = root / "P0001" / "HRTF" / "HRTF" / "44kHz" / "P0001_FreeFieldComp_44kHz.sofa"
pp2_path = root / "P0002" / "HRTF" / "HRTF" / "44kHz" / "P0002_FreeFieldComp_44kHz.sofa"

# Stop early if either expected file is not available.
for path in (pp1_path, pp2_path):
    if not path.exists():
        raise FileNotFoundError(f"Expected SOFA file was not found: {path}")

print(pp1_path)
print(pp2_path)

## Load HRTFs for plotting

{mod}`hrtfpykit.plots` functions operate on loaded {class}`HRTF <hrtfpykit.hrtf.HRTF>` objects. The two SONICOM files use the same measured source grid and sample rate, which makes them suitable for comparison plots and difference plots that require matching positions.

In [ ]:
from hrtfpykit.hrtf import load_hrtf

# Load each measured SOFA file as an HRTF object.
hrtf_p0001 = load_hrtf(pp1_path)
hrtf_p0002 = load_hrtf(pp2_path)

# Keep the comparison objects and labels together.
comparison_hrtfs = [hrtf_p0001, hrtf_p0002]
comparison_labels = ["SONICOM P0001", "SONICOM P0002"]
line_styles = ["-", "--"]

print("P0001 IR shape:", hrtf_p0001.IR.values.shape)
print("P0002 IR shape:", hrtf_p0002.IR.values.shape)
print("P0001 TF shape:", hrtf_p0001.TF.values.shape)
print("P0002 TF shape:", hrtf_p0002.TF.values.shape)

## Import plotting functions

The rest of the tutorial uses one plotting function or one tightly related pair of functions per code cell. Importing the plotting functions once keeps the following examples focused on the plotting call itself, so each cell is easier to run, edit, and inspect independently.

In [ ]:
import numpy as np

from hrtfpykit.hrtf import sht, sht_error, sht_inverse
from hrtfpykit.plots import (
    compare_absolute_ild,
    compare_absolute_itd,
    compare_amplitude,
    compare_magnitude,
    compare_ild,
    compare_ild_difference,
    compare_itd,
    compare_itd_difference,
    compare_lsd,
    plot_absolute_ild,
    plot_absolute_itd,
    plot_amplitude,
    plot_elevation_spectrum,
    plot_etc,
    plot_etc_plane,
    plot_magnitude,
    plot_plane_grid,
    plot_ild,
    plot_ild_fd,
    plot_itd,
    plot_source_grid,
    plot_spectrum_plane,
    sht_reconstruction_comparison,
    sht_reconstruction_error,
)


## Control figure display and text

Most functions in {mod}`hrtfpykit.plots` display their Matplotlib figure immediately. Passing `show=False` creates and configures the figure without opening a window. Passing `show_titles=False` removes generated titles when the caption or surrounding text should describe the figure instead.


In [ ]:
import matplotlib.pyplot as plt

# Create a controlled comparison figure without displaying it immediately.
compare_magnitude(
    comparison_hrtfs,
    positions="front",
    ear="left",
    x_axis="log",
    unit="db",
    reference="max",
    legends=comparison_labels,
    line_colors=["tab:blue", "tab:orange"],
    line_styles=line_styles,
    freq_max=16000.0,
    show=False,
    show_titles=False,
)

# Display the controlled figure when the cell is ready.
plt.show()

## Plot source grids and spatial planes

{func}`plot_source_grid <hrtfpykit.plots.plot_source_grid>` visualizes the current source grid of one loaded HRTF object. {func}`plot_plane_grid <hrtfpykit.plots.plot_plane_grid>` highlights canonical horizontal, median, or frontal planes. These functions read the current {attr}`Sources <hrtfpykit.hrtf.HRTF.Sources>` state, so selected HRTF objects plot their selected grid rather than the original full grid.


In [ ]:
# Plot the full source grid attached to one loaded HRTF.
plot_source_grid(hrtf_p0001, show=False)

# Plot the source grid with canonical spatial planes highlighted.
plot_plane_grid(
    hrtf_p0001,
    plane=["horizontal", "median", "frontal"],
    show=False,
)


## Plot amplitude and magnitude responses

{func}`plot_amplitude <hrtfpykit.plots.plot_amplitude>` shows time-domain HRIR waveforms. {func}`plot_magnitude <hrtfpykit.plots.plot_magnitude>` shows frequency-domain HRTF magnitudes. Both functions use the current {attr}`IR <hrtfpykit.hrtf.HRTF.IR>` and {attr}`TF <hrtfpykit.hrtf.HRTF.TF>` state of the HRTF object passed as the first argument.


In [ ]:
plot_amplitude(
    hrtf_p0001,
    positions=["front", "left"],
    ear="both",
    x_axis="samples",
    show=False,
)

plot_magnitude(
    hrtf_p0001,
    positions=["front", "left"],
    ear="left",
    x_axis="log",
    reference="max",
    show=False,
)


## Plot energy time curves

{func}`plot_etc <hrtfpykit.plots.plot_etc>` shows sample-wise HRIR level in decibels for selected source positions. {func}`plot_etc_plane <hrtfpykit.plots.plot_etc_plane>` shows the same time-domain level view as a heatmap over a measured horizontal or median plane. These plots reflect selections, transforms, padding, gain changes, and sample-rate changes already applied to the HRTF object.


In [ ]:
plot_etc(
    hrtf_p0001,
    positions=["front", "left"],
    ear="both",
    x_axis="samples",
    reference="max",
    show=False,
)

plot_etc_plane(
    hrtf_p0001,
    plane="horizontal",
    plane_angle=0.0,
    ear="left",
    x_axis="samples",
    reference="max",
    show=False,
)


## Plot spectral planes

{func}`plot_spectrum_plane <hrtfpykit.plots.plot_spectrum_plane>` shows HRTF magnitude over a measured horizontal or median plane. {func}`plot_elevation_spectrum <hrtfpykit.plots.plot_elevation_spectrum>` shows how the magnitude response changes across elevation for a fixed azimuth slice.


In [ ]:
plot_spectrum_plane(
    hrtf_p0001,
    plane="horizontal",
    plane_angle=0.0,
    ear="left",
    x_axis="linear",
    show=False,
)

plot_elevation_spectrum(
    hrtf_p0001,
    azimuth="front",
    ear="left",
    show=False,
)


## Plot ITD and ILD cues

Cue plots expose binaural timing and level differences around the source grid. {func}`plot_itd <hrtfpykit.plots.plot_itd>` and {func}`plot_ild <hrtfpykit.plots.plot_ild>` show signed curves over a horizontal plane. {func}`plot_absolute_itd <hrtfpykit.plots.plot_absolute_itd>` shows absolute ITD in polar form, and {func}`plot_absolute_ild <hrtfpykit.plots.plot_absolute_ild>` shows unsigned broad-band ILD in polar form. {func}`plot_ild_fd <hrtfpykit.plots.plot_ild_fd>` shows frequency-dependent ILD over a measured plane.


In [ ]:
plot_itd(hrtf_p0001, plane_angle=0.0, show=False)
plot_absolute_itd(hrtf_p0001, plane_angle=0.0, show=False)

plot_ild(hrtf_p0001, plane_angle=0.0, show=False)
plot_absolute_ild(hrtf_p0001, plane_angle=0.0, show=False)
plot_ild_fd(
    hrtf_p0001,
    plane="horizontal",
    plane_angle=0.0,
    ear="left",
    frequency=4000.0,
    show=False,
)


## Compare amplitude responses

{func}`compare_amplitude <hrtfpykit.plots.compare_amplitude>` overlays HRIR waveforms from multiple HRTF objects at the same requested source direction. This is the time-domain view, so it is useful for inspecting arrival timing, onset shape, early reflections, padding, and waveform differences before moving to frequency-domain interpretation.

The example below compares the left-ear waveform for the resolved `front` position and uses sample indices on the x-axis. Keep this cell small on purpose: users can change the ear, position query, or x-axis and immediately inspect one figure without rerunning unrelated plots.

In [ ]:
# Compare front-direction HRIR waveforms for the left ear.
compare_amplitude(
    comparison_hrtfs,
    positions="front",
    ear="left",
    x_axis="samples",
    legends=comparison_labels,
    line_styles=line_styles,
)

## Compare magnitude responses

{func}`compare_magnitude <hrtfpykit.plots.compare_magnitude>` overlays HRTF magnitude spectra from multiple HRTF objects. This is the frequency-domain view, so it is useful for inspecting spectral notches, broadband gain differences, high-frequency structure, and subject-to-subject variation at specific source directions.

The example compares the left ear at `front` and `back`, uses a logarithmic frequency axis, normalizes dB values with `reference="max"`, and limits the plot to 16 kHz. Named position queries are resolved on each HRTF independently; if the resolved real source positions do not match, hrtfpykit warns because the comparison is no longer perfectly source-aligned.

In [ ]:
# Compare left-ear HRTF magnitudes at two named directions.
compare_magnitude(
    comparison_hrtfs,
    positions=["front", "back"],
    ear="left",
    x_axis="log",
    unit="db",
    reference="max",
    legends=comparison_labels,
    line_styles=line_styles,
    freq_max=16000.0,
)

## Compare absolute and signed ITD cues

ITD comparison plots describe left-right timing cues around the horizontal plane. They are useful for checking whether two subjects or processing outputs preserve similar timing behavior, especially around lateral directions where ITD usually changes strongly.

This cell keeps the ITD diagnostics together: {func}`compare_absolute_itd <hrtfpykit.plots.compare_absolute_itd>` overlays absolute ITD magnitude on a polar axis, while {func}`compare_itd <hrtfpykit.plots.compare_itd>` overlays signed ITD values across azimuth. The requested `plane_angle=0.0` selects the nearest measured horizontal plane in each HRTF.

In [ ]:
# Compare absolute ITD magnitude on the horizontal plane.
compare_absolute_itd(
    comparison_hrtfs,
    plane_angle=0.0,
    legends=comparison_labels,
    line_styles=line_styles,
)

# Compare signed ITD across azimuth.
compare_itd(
    comparison_hrtfs,
    plane_angle=0.0,
    legends=comparison_labels,
    line_styles=line_styles,
)

## Compare unsigned and signed broad-band ILD cues

Broad-band ILD comparison plots describe left-right level cues around the horizontal plane. They are useful for inspecting directional shadowing behavior and subject-to-subject differences that are not visible from one single-ear magnitude curve.

This cell keeps the broad-band ILD diagnostics together: {func}`compare_absolute_ild <hrtfpykit.plots.compare_absolute_ild>` overlays absolute broad-band ILD on a polar axis, while {func}`compare_ild <hrtfpykit.plots.compare_ild>` overlays signed ILD values across azimuth.

In [ ]:
# Compare unsigned broad-band ILD on the horizontal plane.
compare_absolute_ild(
    comparison_hrtfs,
    plane_angle=0.0,
    legends=comparison_labels,
    line_styles=line_styles,
)

# Compare signed ILD across azimuth.
compare_ild(
    comparison_hrtfs,
    plane_angle=0.0,
    legends=comparison_labels,
    line_styles=line_styles,
)

## Compare ITD difference over the source grid

{func}`compare_itd_difference <hrtfpykit.plots.compare_itd_difference>` compares a reference HRTF against one or more HRTFs and maps the ITD difference at every shared source position. By default it plots absolute differences. Set `absolute=False` when the sign of `compared - reference` is needed.

The example below reports the difference in samples and uses `azimuth_range_mode="-180-180"` so front-facing directions are easier to read around zero azimuth.


In [ ]:
# Compare ITD difference over the source grid.
compare_itd_difference(
    hrtf_p0001,
    hrtf_p0002,
    output="samples",
    absolute=True,
    azimuth_range_mode="-180-180",
    colormap="viridis",
)


## Compare broad-band ILD difference over the source grid

{func}`compare_ild_difference <hrtfpykit.plots.compare_ild_difference>` compares a reference HRTF against one or more HRTFs and maps one broad-band ILD difference value per shared source position. By default it plots absolute differences in dB. Set `absolute=False` when the sign of `compared - reference` is needed.

The plot is broad-band only because the source grid needs one scalar value per position. Frequency-dependent ILD differences are source and frequency arrays and should be inspected with a frequency plot or the {func}`ild_difference <hrtfpykit.hrtf.ild_difference>` metric.


In [ ]:
# Compare broad-band ILD difference over the source grid.
compare_ild_difference(
    hrtf_p0001,
    hrtf_p0002,
    absolute=True,
    azimuth_range_mode="-180-180",
    colormap="magma",
)


## Compare frequency-reduced LSD over the source grid

{func}`compare_lsd <hrtfpykit.plots.compare_lsd>` compares a reference HRTF against one or more HRTFs and plots one frequency-reduced spectral-distance value per source position. If several compared HRTFs are provided, their LSD maps are reduced with `reduction_method` before plotting.

The example below compares the left ear and keeps the result as a spatial map. A high-value region means the two HRTFs are spectrally less similar at those source directions.


In [ ]:
# Compare one frequency-averaged LSD value per source position.
compare_lsd(
    hrtf_p0001,
    hrtf_p0002,
    ear="left",
    azimuth_range_mode="-180-180",
    colormap="viridis",
)


## Prepare spherical-harmonic reconstruction data

The spherical-harmonic plotting functions diagnose SHT reconstruction quality. Before plotting, compute a spherical-harmonic representation with {func}`sht <hrtfpykit.hrtf.sht>`, reconstruct magnitudes on the original source grid with {func}`sht_inverse <hrtfpykit.hrtf.sht_inverse>`, and compute summary errors with {func}`sht_error <hrtfpykit.hrtf.sht_error>`.

This preparation cell creates the data used by the next two plotting cells. The reconstruction contains magnitude only; it does not reconstruct phase and it does not create a new complex HRTF object.

In [ ]:
# Build a low-order spherical-harmonic magnitude representation for both ears.
sh = sht(hrtf_p0001, sh_order=4, ear="both")
reconstructed_magnitude = sht_inverse(sh)

# Compare reconstructed magnitudes with the original linear magnitudes.
original_magnitude = np.abs(hrtf_p0001.TF.values[:, 0:2, :])
abs_err, rel_err, rms_err, max_err = sht_error(
    original_magnitude=original_magnitude,
    reconstructed_magnitude=reconstructed_magnitude,
    magnitude="db",
    reference="max",
)

print("SH coefficients shape:", sh.C.shape)
print("Reconstructed magnitude shape:", reconstructed_magnitude.shape)
print("Global RMS reconstruction error (dB):", f"{rms_err:.2f}")
print("Maximum reconstruction error (dB):", f"{max_err:.2f}")

## Compare original and SH-reconstructed spectra

{func}`sht_reconstruction_comparison <hrtfpykit.plots.sht_reconstruction_comparison>` overlays the original HRTF magnitude and the SH-reconstructed magnitude for one source direction and ear. This is useful for seeing whether the chosen SH order preserves the spectral shape at a specific direction, instead of relying only on global error metrics.

In [ ]:
# Overlay original and reconstructed spectra at one direction.
sht_reconstruction_comparison(
    hrtf_p0001,
    reconstructed_magnitude,
    position="front",
    ear="left",
    x_axis="log",
    unit="db",
    reference="max",
    freq_max=16000.0,
)

## Plot SH reconstruction error

{func}`sht_reconstruction_error <hrtfpykit.plots.sht_reconstruction_error>` plots the point-wise reconstruction error for one source direction and ear. This is the most direct view when the question is where the SH approximation loses spectral detail across frequency.

In [ ]:
# Plot the point-wise reconstruction error for the same direction.
sht_reconstruction_error(
    hrtf_p0001,
    reconstructed_magnitude,
    position="front",
    ear="left",
    x_axis="log",
    magnitude="db",
    reference="max",
    freq_max=16000.0,
)